# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

## Prediksi Dropout Siswa - Jaya Jaya Institut

- **Nama:** Failasuf Indi M
- **Email:** failasufindi123@gmail.com
- **Id Dicoding:** failasuf

## Business Understanding

### Latar Belakang
**Jaya Jaya Institut** merupakan salah satu institusi pendidikan perguruan yang telah berdiri sejak tahun 2000. Hingga saat ini ia telah mencetak banyak lulusan dengan reputasi yang sangat baik. Akan tetapi, terdapat banyak juga siswa yang tidak menyelesaikan pendidikannya alias **dropout**.

Jumlah dropout yang tinggi ini tentunya menjadi salah satu masalah yang besar untuk sebuah institusi pendidikan. Oleh karena itu, Jaya Jaya Institut ingin **mendeteksi secepat mungkin** siswa yang mungkin akan melakukan dropout sehingga dapat diberi bimbingan khusus.

### Permasalahan Bisnis
1. Bagaimana **mengidentifikasi faktor-faktor utama** yang menyebabkan siswa melakukan dropout?
2. Bagaimana membangun **model machine learning** yang dapat memprediksi apakah seorang siswa akan **Dropout** atau **Graduate** secara akurat?
3. Bagaimana membuat **sistem monitoring** melalui dashboard agar pihak institusi dapat memantau performa siswa secara real-time?

### Cakupan Proyek
1. **Analisis Data (EDA):** Eksplorasi dataset siswa untuk menemukan pola dan insight terkait dropout.
2. **Preprocessing & Feature Engineering:** Menyiapkan data untuk pemodelan, termasuk filtering hanya Dropout & Graduate, dan handling class imbalance.
3. **Modeling:** Melatih dan membandingkan beberapa model binary classification (Logistic Regression, Random Forest, Gradient Boosting).
4. **Evaluasi:** Mengevaluasi performa model dengan metrik yang relevan (Accuracy, F1, Recall Dropout, ROC-AUC).
5. **Dashboard:** Membuat business dashboard menggunakan Google Looker Studio.
6. **Prototype:** Membangun prototype prediksi menggunakan Streamlit dan men-deploy ke Streamlit Community Cloud.

## Persiapan

### Menyiapkan library yang dibutuhkan

Berikut adalah library utama yang digunakan:
- **pandas & numpy**: Manipulasi dan analisis data
- **matplotlib & seaborn**: Visualisasi data
- **scikit-learn**: Preprocessing, modeling, dan evaluasi
- **imbalanced-learn**: Handling class imbalance dengan SMOTE
- **joblib**: Menyimpan dan memuat model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print("Library berhasil dimuat!")

### Menyiapkan data yang akan digunakan

Dataset diambil dari repository Dicoding yang berisi data performa siswa di berbagai program studi. Dataset ini berisi informasi yang diketahui pada saat pendaftaran siswa (jalur akademik, demografis, dan faktor sosial-ekonomi) serta performa akademik siswa pada akhir semester 1 dan 2.

In [ ]:
# Load dataset
df_full = pd.read_csv('data.csv', delimiter=';')
print(f"Dataset berhasil dimuat!")
print(f"Jumlah baris: {df_full.shape[0]:,}")
print(f"Jumlah kolom: {df_full.shape[1]}")
print(f"\nDistribusi Status (full dataset):")
for status, count in df_full['Status'].value_counts().items():
    print(f"  {status}: {count} ({count/len(df_full)*100:.1f}%)")

In [ ]:
# Melihat 5 baris pertama
df_full.head()

In [ ]:
# Info dataset
df_full.info()

## Data Understanding

Pada tahap ini, kita akan melakukan eksplorasi data untuk memahami karakteristik dataset dan menemukan insight yang berguna.

### Pemeriksaan Kualitas Data

In [ ]:
# Cek missing values dan duplikat
print("=" * 50)
print("PEMERIKSAAN KUALITAS DATA")
print("=" * 50)
print(f"\nMissing values per kolom:")
missing = df_full.isnull().sum()
if missing.sum() == 0:
    print("  Tidak ada missing values! ✓")
else:
    print(missing[missing > 0])

print(f"\nJumlah baris duplikat: {df_full.duplicated().sum()}")
print(f"\nStatistik deskriptif:")
df_full.describe().round(2)

### Distribusi Target (Status Siswa)

Kolom target `Status` memiliki 3 kategori: **Dropout**, **Enrolled**, dan **Graduate**. Untuk keperluan pemodelan, hanya siswa berstatus **Dropout** dan **Graduate** yang akan digunakan, karena tujuan sistem adalah memprediksi apakah seorang siswa akan berpotensi dropout atau lulus. Siswa **Enrolled** belum memiliki label akhir sehingga dipisahkan.

In [ ]:
# Distribusi target pada full dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#ff6b6b', '#f7b731', '#26de81']
target_counts = df_full['Status'].value_counts()
ax1 = axes[0]
bars = ax1.bar(target_counts.index, target_counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Distribusi Status Siswa (Full Dataset)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Jumlah Siswa')
for bar, val in zip(bars, target_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
             f'{val}\n({val/len(df_full)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(target_counts.values, labels=target_counts.index,
                                     colors=colors, autopct='%1.1f%%', startangle=90,
                                     explode=(0.05, 0.05, 0.05), shadow=True,
                                     textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax2.set_title('Proporsi Status Siswa', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTotal siswa: {len(df_full):,}")
print(f"Siswa Enrolled yang akan dipisahkan: {target_counts.get('Enrolled', 0)}")
print(f"Siswa yang digunakan untuk modeling (Dropout + Graduate): {target_counts.get('Dropout',0) + target_counts.get('Graduate',0)}")

### Analisis Performa Akademik vs Status

Performa akademik (jumlah mata kuliah yang disetujui dan nilai rata-rata) merupakan indikator kunci dalam memprediksi dropout.

In [ ]:
# Performa akademik vs Status (hanya Dropout & Graduate)
df_model_eda = df_full[df_full['Status'].isin(['Dropout', 'Graduate'])].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
status_order = ['Dropout', 'Graduate']
colors_status = {'Dropout': '#ff6b6b', 'Graduate': '#26de81'}

# 1. Units approved sem 1
ax = axes[0, 0]
for status in status_order:
    data = df_model_eda[df_model_eda['Status'] == status]['Curricular_units_1st_sem_approved']
    ax.hist(data, bins=20, alpha=0.6, label=status, color=colors_status[status], edgecolor='white')
ax.set_title('Distribusi Mata Kuliah Disetujui (Sem 1)', fontsize=12, fontweight='bold')
ax.set_xlabel('Jumlah Mata Kuliah Approved')
ax.legend()

# 2. Grade sem 1
ax = axes[0, 1]
sns.boxplot(data=df_model_eda, x='Status', y='Curricular_units_1st_sem_grade',
            order=status_order, palette=colors_status, ax=ax)
ax.set_title('Distribusi Nilai Rata-rata Semester 1', fontsize=12, fontweight='bold')

# 3. Units approved sem 2
ax = axes[1, 0]
for status in status_order:
    data = df_model_eda[df_model_eda['Status'] == status]['Curricular_units_2nd_sem_approved']
    ax.hist(data, bins=20, alpha=0.6, label=status, color=colors_status[status], edgecolor='white')
ax.set_title('Distribusi Mata Kuliah Disetujui (Sem 2)', fontsize=12, fontweight='bold')
ax.set_xlabel('Jumlah Mata Kuliah Approved')
ax.legend()

# 4. Grade sem 2
ax = axes[1, 1]
sns.boxplot(data=df_model_eda, x='Status', y='Curricular_units_2nd_sem_grade',
            order=status_order, palette=colors_status, ax=ax)
ax.set_title('Distribusi Nilai Rata-rata Semester 2', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print means
print("\nRata-rata Performa Akademik per Status (Dropout vs Graduate):")
academic_cols = ['Curricular_units_1st_sem_approved', 'Curricular_units_1st_sem_grade',
                 'Curricular_units_2nd_sem_approved', 'Curricular_units_2nd_sem_grade']
print(df_model_eda.groupby('Status')[academic_cols].mean().round(2).to_string())
print("\nInsight: Siswa Dropout memiliki rata-rata performa akademik yang jauh lebih rendah")
print("dibandingkan Graduate, terutama pada jumlah mata kuliah yang approved dan nilai rata-rata.")

### Analisis Faktor Finansial vs Status

Faktor finansial (status pembayaran, beasiswa, status debtor) juga berpengaruh terhadap risiko dropout.

In [ ]:
# Faktor finansial vs Status (Dropout vs Graduate)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

financial_features = [
    ('Tuition_fees_up_to_date', 'SPP Terbayar'),
    ('Scholarship_holder', 'Penerima Beasiswa'),
    ('Debtor', 'Status Debtor')
]
status_order = ['Dropout', 'Graduate']

for idx, (feat, title) in enumerate(financial_features):
    ax = axes[idx]
    ct = pd.crosstab(df_model_eda['Status'], df_model_eda[feat], normalize='index') * 100
    ct = ct.loc[status_order]
    ct.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1)
    ax.set_title(f'{title} per Status', fontsize=12, fontweight='bold')
    ax.set_ylabel('Persentase (%)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(['Tidak', 'Ya'], loc='upper right')

plt.tight_layout()
plt.show()

# Print stats
tuition_by_status = df_model_eda.groupby('Status')['Tuition_fees_up_to_date'].mean() * 100
scholar_by_status = df_model_eda.groupby('Status')['Scholarship_holder'].mean() * 100
debtor_by_status = df_model_eda.groupby('Status')['Debtor'].mean() * 100

print("\nInsight Faktor Finansial:")
print(f"- SPP terbayar: Dropout={tuition_by_status['Dropout']:.1f}%, Graduate={tuition_by_status['Graduate']:.1f}%")
print(f"- Penerima beasiswa: Dropout={scholar_by_status['Dropout']:.1f}%, Graduate={scholar_by_status['Graduate']:.1f}%")
print(f"- Status debtor: Dropout={debtor_by_status['Dropout']:.1f}%, Graduate={debtor_by_status['Graduate']:.1f}%")
print("=> Faktor finansial sangat berkorelasi dengan risiko dropout!")

### Analisis Usia dan Dropout Rate per Program Studi

In [ ]:
# Usia dan Course analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
status_order = ['Dropout', 'Graduate']
colors_status = {'Dropout': '#ff6b6b', 'Graduate': '#26de81'}

# 1. Age distribution
ax = axes[0]
for status in status_order:
    data = df_model_eda[df_model_eda['Status'] == status]['Age_at_enrollment']
    ax.hist(data, bins=30, alpha=0.6, label=status, color=colors_status[status], edgecolor='white')
ax.set_title('Distribusi Usia Saat Mendaftar (Dropout vs Graduate)', fontsize=12, fontweight='bold')
ax.set_xlabel('Usia')
ax.legend()

# 2. Dropout rate per course (dari full dataset)
ax = axes[1]
course_names = {
    33: "Biofuel Tech", 171: "Animation", 8014: "Social Service (eve)",
    9003: "Agronomy", 9070: "Comm Design", 9085: "Vet. Nursing",
    9119: "Informatics", 9130: "Equinculture", 9147: "Management",
    9238: "Social Service", 9254: "Tourism", 9500: "Nursing",
    9556: "Oral Hygiene", 9670: "Ads & Marketing", 9773: "Journalism",
    9853: "Basic Education", 9991: "Management (eve)"
}

dropout_by_course = df_full.groupby('Course').apply(
    lambda x: (x['Status'] == 'Dropout').sum() / len(x) * 100
).sort_values(ascending=True)

avg_dropout_rate = (df_full['Status'] == 'Dropout').mean() * 100
course_labels = [course_names.get(c, str(c)) for c in dropout_by_course.index]
colors_bar = ['#ff6b6b' if v > 40 else '#f7b731' if v > 25 else '#26de81' for v in dropout_by_course.values]
ax.barh(course_labels, dropout_by_course.values, color=colors_bar, edgecolor='white')
ax.set_title('Dropout Rate per Program Studi', fontsize=12, fontweight='bold')
ax.set_xlabel('Dropout Rate (%)')
ax.axvline(x=avg_dropout_rate, color='gray', linestyle='--', alpha=0.7,
           label=f'Rata-rata ({avg_dropout_rate:.1f}%)')
ax.legend()

plt.tight_layout()
plt.show()

print("\nInsight:")
print(f"- Rata-rata usia Dropout: {df_model_eda[df_model_eda['Status']=='Dropout']['Age_at_enrollment'].mean():.1f} tahun")
print(f"- Rata-rata usia Graduate: {df_model_eda[df_model_eda['Status']=='Graduate']['Age_at_enrollment'].mean():.1f} tahun")
print(f"- Dropout rate keseluruhan: {avg_dropout_rate:.1f}%")
print("- Program dengan dropout rate tinggi memerlukan perhatian khusus")

### Korelasi antar Fitur Numerik

Heatmap korelasi membantu mengidentifikasi hubungan linear antar fitur.

In [ ]:
# Correlation heatmap (selected features)
selected_cols = [
    'Admission_grade', 'Previous_qualification_grade',
    'Curricular_units_1st_sem_approved', 'Curricular_units_1st_sem_grade',
    'Curricular_units_2nd_sem_approved', 'Curricular_units_2nd_sem_grade',
    'Tuition_fees_up_to_date', 'Scholarship_holder', 'Debtor',
    'Age_at_enrollment', 'Unemployment_rate', 'GDP'
]

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df_model_eda[selected_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Fitur Terpilih)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInsight: Terdapat korelasi positif kuat antara units approved sem 1 & sem 2,")
print("serta antara grade sem 1 & sem 2, menunjukkan konsistensi performa akademik.")

## Data Preparation / Preprocessing

Pada tahap ini kita melakukan beberapa langkah preprocessing sesuai ketentuan:

1. **Filtering Data**: Hanya siswa berstatus **Dropout** dan **Graduate** yang digunakan untuk pemodelan. Siswa **Enrolled** dipisahkan karena belum memiliki label akhir.
2. **Binary Target Encoding**: Target dijadikan binary — `1 = Dropout`, `0 = Graduate`
3. **Feature Engineering** - membuat fitur baru yang bermakna
4. **Feature Scaling** menggunakan StandardScaler
5. **Train-Test Split** dengan stratified sampling
6. **SMOTE** untuk mengatasi class imbalance pada data training

In [ ]:
# 1. Pisahkan siswa Enrolled (belum memiliki label akhir)
df_enrolled = df_full[df_full['Status'] == 'Enrolled'].copy()
print(f"Siswa Enrolled (dipisahkan): {len(df_enrolled)}")
print("=> Data ini disimpan sebagai 'enrolled_data.csv' untuk prediksi masa depan")

# Simpan enrolled data
df_enrolled.to_csv('enrolled_data.csv', index=False)
print(f"   enrolled_data.csv berhasil disimpan!")

# 2. Dataset untuk modeling: hanya Dropout dan Graduate
df = df_full[df_full['Status'].isin(['Dropout', 'Graduate'])].copy()
print(f"\nDataset untuk modeling: {df.shape}")
print(f"Distribusi Status:")
for status, count in df['Status'].value_counts().items():
    print(f"  {status}: {count} ({count/len(df)*100:.1f}%)")

In [ ]:
# 3. Binary Target Encoding
# 1 = Dropout, 0 = Graduate
df['Target'] = df['Status'].map({'Dropout': 1, 'Graduate': 0})

print("Binary Target Encoding:")
print("  Dropout  → 1")
print("  Graduate → 0")
print(f"\nDistribusi target:")
print(f"  Dropout (1): {df['Target'].sum()} siswa")
print(f"  Graduate (0): {(df['Target'] == 0).sum()} siswa")

In [ ]:
# 4. Feature Engineering
print("Membuat fitur baru...")
df['Sem1_approval_rate'] = np.where(
    df['Curricular_units_1st_sem_enrolled'] > 0,
    df['Curricular_units_1st_sem_approved'] / df['Curricular_units_1st_sem_enrolled'],
    0
)
df['Sem2_approval_rate'] = np.where(
    df['Curricular_units_2nd_sem_enrolled'] > 0,
    df['Curricular_units_2nd_sem_approved'] / df['Curricular_units_2nd_sem_enrolled'],
    0
)
df['Total_approved'] = df['Curricular_units_1st_sem_approved'] + df['Curricular_units_2nd_sem_approved']
df['Avg_grade'] = (df['Curricular_units_1st_sem_grade'] + df['Curricular_units_2nd_sem_grade']) / 2

print("Fitur baru berhasil dibuat:")
print("  - Sem1_approval_rate: Rasio mata kuliah approved/enrolled semester 1")
print("  - Sem2_approval_rate: Rasio mata kuliah approved/enrolled semester 2")
print("  - Total_approved: Total mata kuliah yang approved (sem 1 + sem 2)")
print("  - Avg_grade: Rata-rata nilai semester 1 dan 2")

In [ ]:
# 5. Definisi fitur
feature_cols = [
    'Marital_status', 'Application_mode', 'Application_order', 'Course',
    'Daytime_evening_attendance', 'Previous_qualification', 'Previous_qualification_grade',
    'Nacionality', 'Mothers_qualification', 'Fathers_qualification',
    'Mothers_occupation', 'Fathers_occupation', 'Admission_grade',
    'Displaced', 'Educational_special_needs', 'Debtor',
    'Tuition_fees_up_to_date', 'Gender', 'Scholarship_holder',
    'Age_at_enrollment', 'International',
    'Curricular_units_1st_sem_credited', 'Curricular_units_1st_sem_enrolled',
    'Curricular_units_1st_sem_evaluations', 'Curricular_units_1st_sem_approved',
    'Curricular_units_1st_sem_grade', 'Curricular_units_1st_sem_without_evaluations',
    'Curricular_units_2nd_sem_credited', 'Curricular_units_2nd_sem_enrolled',
    'Curricular_units_2nd_sem_evaluations', 'Curricular_units_2nd_sem_approved',
    'Curricular_units_2nd_sem_grade', 'Curricular_units_2nd_sem_without_evaluations',
    'Unemployment_rate', 'Inflation_rate', 'GDP',
    # Engineered features
    'Sem1_approval_rate', 'Sem2_approval_rate', 'Total_approved', 'Avg_grade'
]

X = df[feature_cols]
y = df['Target']
print(f"Jumlah fitur: {len(feature_cols)}")
print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

# 6. Train-Test Split (80:20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain-Test Split (80:20, Stratified):")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: Dropout={y_train.sum()}, Graduate={(y_train==0).sum()}")
print(f"  y_test:  Dropout={y_test.sum()}, Graduate={(y_test==0).sum()}")

# 7. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\nFeature scaling dengan StandardScaler berhasil diterapkan.")

# 8. SMOTE untuk class imbalance (hanya pada data training)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"\nSebelum SMOTE: {X_train_scaled.shape[0]} samples")
print(f"Setelah SMOTE: {X_train_resampled.shape[0]} samples")
do_after = y_train_resampled.sum()
gr_after = len(y_train_resampled) - do_after
print(f"Distribusi setelah SMOTE: Dropout={do_after}, Graduate={gr_after}")

**Catatan penting:**
- Data untuk pemodelan **hanya mencakup siswa Dropout dan Graduate**. Siswa Enrolled dipisahkan karena belum memiliki label akhir dan tidak layak digunakan sebagai data training.
- Target bersifat **binary**: `1 = Dropout`, `0 = Graduate`
- SMOTE hanya diterapkan pada **data training**, bukan data testing. Hal ini untuk menghindari data leakage.
- StandardScaler di-fit pada data training dan hanya di-transform pada data testing.

## Modeling

Kita akan melatih dan membandingkan 3 model **binary classification**:
1. **Logistic Regression** - sebagai baseline model
2. **Random Forest** - ensemble model yang robust dan interpretable
3. **Gradient Boosting** - model boosting dengan performa tinggi

Kemudian model terbaik akan di-tune hyperparameter-nya menggunakan GridSearchCV.

In [ ]:
# Training 3 model binary classification
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    
    # Recall khusus kelas Dropout (class 1)
    recall_dropout = recall_score(y_test, y_pred, pos_label=1, average='binary')
    
    results[name] = {
        'model': model,
        'accuracy': acc,
        'f1_weighted': f1,
        'recall_weighted': recall,
        'precision_weighted': precision,
        'recall_dropout': recall_dropout,
        'y_pred': y_pred,
    }
    print(f"  Accuracy: {acc:.4f} | F1: {f1:.4f} | Recall Dropout: {recall_dropout:.4f}")
    print()

# Tabel perbandingan
print("\n" + "=" * 70)
print("PERBANDINGAN MODEL (Binary: Dropout vs Graduate)")
print("=" * 70)
comparison_data = []
for name, r in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{r['accuracy']:.4f}",
        'F1 (Weighted)': f"{r['f1_weighted']:.4f}",
        'Precision': f"{r['precision_weighted']:.4f}",
        'Recall': f"{r['recall_weighted']:.4f}",
        'Recall Dropout': f"{r['recall_dropout']:.4f}",
    })
print(pd.DataFrame(comparison_data).to_string(index=False))

### Hyperparameter Tuning

Model terbaik dipilih berdasarkan kombinasi **F1-Score (weighted)** dan **Recall kelas Dropout**, kemudian dilakukan hyperparameter tuning menggunakan GridSearchCV dengan 5-fold cross-validation.

In [ ]:
# Pilih model terbaik
best_name = max(results, key=lambda k: results[k]['f1_weighted'] * 0.5 + results[k]['recall_dropout'] * 0.5)
print(f"Model terbaik (sebelum tuning): {best_name}")

# Hyperparameter tuning
if 'Random Forest' in best_name:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
    }
    grid_model = RandomForestClassifier(random_state=42, n_jobs=-1)
elif 'Gradient Boosting' in best_name:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
        'min_samples_split': [2, 5],
    }
    grid_model = GradientBoostingClassifier(random_state=42)
else:
    param_grid = {
        'C': [0.1, 1, 10],
        'solver': ['lbfgs', 'saga'],
    }
    grid_model = LogisticRegression(max_iter=1000, random_state=42)

print(f"\nMenjalankan GridSearchCV...")
grid_search = GridSearchCV(
    grid_model, param_grid, cv=5, scoring='f1_weighted',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_resampled, y_train_resampled)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

## Evaluation

Evaluasi model final dilakukan pada **data test** yang belum pernah dilihat oleh model selama training. Metrik yang digunakan:
- **Accuracy**: Persentase prediksi benar secara keseluruhan
- **F1-Score**: Harmonic mean dari precision dan recall
- **Recall (Dropout)**: Sangat penting — kemampuan model mendeteksi siswa yang benar-benar dropout
- **ROC-AUC**: Kemampuan diskriminasi model secara keseluruhan (binary)
- **Confusion Matrix**: Visualisasi detail prediksi benar dan salah per kelas

In [ ]:
# Evaluasi model final
y_pred_final = best_model.predict(X_test_scaled)
y_proba_final = best_model.predict_proba(X_test_scaled)[:, 1]  # prob Dropout (class 1)

acc = accuracy_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final, average='weighted')
recall_do = recall_score(y_test, y_pred_final, pos_label=1, average='binary')

print("=" * 60)
print(f"EVALUASI MODEL FINAL: {best_name} (Tuned)")
print("Binary Classification: Dropout (1) vs Graduate (0)")
print("=" * 60)
print(f"\nAccuracy:         {acc:.4f}")
print(f"F1 (Weighted):    {f1:.4f}")
print(f"Recall Dropout:   {recall_do:.4f}")

# ROC AUC (binary)
try:
    roc_auc = roc_auc_score(y_test, y_proba_final)
    print(f"ROC-AUC (binary): {roc_auc:.4f}")
except:
    roc_auc = None

print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print('='*60)
print(classification_report(y_test, y_pred_final, target_names=['Graduate (0)', 'Dropout (1)']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Heatmap count
cm = confusion_matrix(y_test, y_pred_final)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Graduate', 'Dropout'],
            yticklabels=['Graduate', 'Dropout'],
            ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix (Count)', fontsize=14, fontweight='bold')

# 2. Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax = axes[1]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=['Graduate', 'Dropout'],
            yticklabels=['Graduate', 'Dropout'],
            ax=ax, linewidths=0.5, linecolor='white',
            annot_kws={'size': 12, 'fontweight': 'bold'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nAnalisis Confusion Matrix:")
print(f"- Graduate: {cm[0][0]} dari {cm[0].sum()} terdeteksi benar ({cm_norm[0][0]*100:.1f}%)")
print(f"- Dropout:  {cm[1][1]} dari {cm[1].sum()} terdeteksi benar ({cm_norm[1][1]*100:.1f}%)")
print(f"- False Negative (Dropout tidak terdeteksi): {cm[1][0]}")

### Feature Importance

Feature importance menunjukkan fitur mana yang paling berpengaruh dalam prediksi model. Informasi ini sangat berguna untuk memahami faktor-faktor utama penyebab dropout.

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Plot top 15
    top15 = feat_imp.head(15)
    fig, ax = plt.subplots(figsize=(10, 8))
    colors_grad = plt.cm.Blues(np.linspace(0.4, 0.9, len(top15)))[::-1]
    bars = ax.barh(range(len(top15)), top15['Importance'].values[::-1], color=colors_grad)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels(top15['Feature'].values[::-1], fontsize=10)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title('Top 15 Feature Importance (Binary: Dropout vs Graduate)', fontsize=14, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, top15['Importance'].values[::-1])):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Feature Importance:")
    for _, row in feat_imp.head(10).iterrows():
        print(f"  {row['Feature']:45s} {row['Importance']:.4f}")
    
    print("\nInsight Kunci:")
    print("- Rasio approval dan grade semester 1 & 2 merupakan prediktor terkuat")
    print("- Status pembayaran SPP (Tuition_fees_up_to_date) juga sangat penting")
    print("- Usia saat mendaftar berperan signifikan dalam prediksi")

### Menyimpan Model dan Artifacts

Model, scaler, dan konfigurasi disimpan menggunakan joblib agar dapat digunakan kembali pada aplikasi Streamlit.

In [ ]:
# Simpan model dan artifacts
import os, json
os.makedirs('model', exist_ok=True)

joblib.dump(best_model, 'model/model.joblib')
joblib.dump(scaler, 'model/scaler.joblib')
joblib.dump(feature_cols, 'model/feature_names.joblib')

# Simpan label mapping
label_mapping = {'Dropout': 1, 'Graduate': 0}
with open('model/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

print("Model dan artifacts berhasil disimpan:")
print("  - model/model.joblib         (model binary classifier)")
print("  - model/scaler.joblib        (StandardScaler)")
print("  - model/feature_names.joblib (daftar fitur)")
print("  - model/label_mapping.json   (mapping: Dropout=1, Graduate=0)")

# Simpan feature importance
if hasattr(best_model, 'feature_importances_'):
    feat_imp.to_csv('model/feature_importance.csv', index=False)
    print("  - model/feature_importance.csv")

print("\nSemua artifacts berhasil disimpan! Siap digunakan di aplikasi Streamlit.")
print(f"\nenrolled_data.csv: Data {len(df_enrolled)} siswa Enrolled tersimpan untuk prediksi masa depan.")

## Kesimpulan

### Temuan Utama

1. **Pendekatan Pemodelan:** Hanya siswa berstatus **Dropout** dan **Graduate** yang digunakan untuk training model (binary classification). Siswa **Enrolled** dipisahkan karena belum memiliki label akhir.

2. **Distribusi Data (Modeling):**
   - Dataset modeling berisi siswa Dropout dan Graduate
   - Target binary: `1 = Dropout`, `0 = Graduate`
   - Ketidakseimbangan kelas ditangani dengan SMOTE pada data training

3. **Faktor Kunci Dropout:**
   - **Performa Akademik** (paling dominan): Siswa dropout rata-rata menyelesaikan jauh lebih sedikit mata kuliah dibanding Graduate. Rasio approval rate semester 1 & 2 menjadi prediktor terkuat.
   - **Faktor Finansial**: Proporsi siswa dropout yang belum melunasi SPP, rendahnya penerimaan beasiswa, dan tingginya status debtor sangat berkorelasi dengan risiko dropout.
   - **Usia**: Rata-rata usia siswa dropout lebih tinggi dari Graduate.
   - **Program Studi**: Beberapa program memiliki dropout rate jauh di atas rata-rata.

4. **Model Machine Learning (Binary):**
   - Model terbaik dipilih melalui perbandingan 3 algoritma dan hyperparameter tuning
   - Model berhasil memprediksi Dropout vs Graduate dengan performa yang baik
   - Engineered features (approval rate, total approved, avg grade) menjadi prediktor terkuat

5. **Sistem Deteksi Dini:** Model ini dapat diintegrasikan sebagai Early Warning System untuk mendeteksi siswa berisiko dropout, sementara data siswa Enrolled dapat diprediksi menggunakan model yang sama di masa mendatang.